In [ ]:
# Load JSON files into DuckDB (works for both NDJSON and regular JSON arrays/objects)
import duckdb
from pathlib import Path

# ---- paths ----
project_root = Path.cwd().resolve()
while project_root.name != "big_data_assignment" and project_root.parent != project_root:
    project_root = project_root.parent

json_dir = project_root / "data" / "raw" / "json"
writers = json_dir / "writing.json"
directers = json_dir / "directing.json"

# DuckDB database file (kept in your members/lisa folder like in your tree)
db_path = project_root / "members" / "lisa" / "imdb.duckdb"

con = duckdb.connect(str(db_path))

# Nice defaults
con.execute("PRAGMA threads=4;")

# ---- create tables from JSON ----
# DuckDB can read:
# - NDJSON (one JSON object per line)
# - JSON arrays/objects (auto=TRUE tries both)
#
# We create tables and insert the parsed rows.
con.execute("""
CREATE OR REPLACE TABLE writing AS
SELECT *
FROM read_json_auto(?)
""", [str(writers_json)])

con.execute("""
CREATE OR REPLACE TABLE directing AS
SELECT *
FROM read_json_auto(?)
""", [str(directing_json)])

# ---- sanity checks ----
print("Tables:", con.execute("SHOW TABLES").fetchall())
print("writing rows:", con.execute("SELECT COUNT(*) FROM writing").fetchone()[0])
print("directing rows:", con.execute("SELECT COUNT(*) FROM directing").fetchone()[0])

# Peek
display(con.execute("SELECT * FROM writing LIMIT 5").df())
display(con.execute("SELECT * FROM directing LIMIT 5").df())

Tables: [('directing',), ('writing',)]
writing rows: 22428
directing rows: 1


,movie,writer
0,tt0003740,nm0195339
1,tt0003740,nm0515385
2,tt0003740,nm0665163
3,tt0003740,nm0758215
4,tt0008663,nm0406585


,movie,director
0,"{'0': 'tt0003740', '1': 'tt0008663', '2': 'tt0...","{'0': 'nm0665163', '1': 'nm0803705', '2': 'nm0..."


### Environment setup

Before running this notebook, install the project dependencies in a terminal from the project root (`big_data_assignment`):

```bash
cd ~/Desktop/Big-Data/big_data_assignment
pip install -r requirements.txt
```

After installing, restart the Jupyter kernel and re-run the first environment cell.

In [2]:
%load_ext autoreload
%autoreload 2

from src.utils.config import load_config
from src.data.dataloaders import load_train_data, load_validation_data
from src.pipeline.baseline import run_baselines


ModuleNotFoundError: No module named 'xgboost'

# Shared baseline notebook

This notebook runs the **shared baseline models** on the IMDB CSV data using the project source code.

- Uses the central `config/config.yaml` to configure **data paths, feature settings, and hyperparameters**.
- Imports shared helpers from `src/` (config loader, dataloaders, and baseline pipeline).
- Works **only on CSV data referenced in the YAML config**.
- **JSON files or other external datasets are _not_ included here**; those should be handled in separate, member-specific notebooks.


## 1. Environment and imports

This cell:
- Locates the project root folder (named `big_data_assignment`).
- Adds `src/` to `sys.path` so we can import the shared modules.
- (Optionally) enables autoreload for a smoother dev experience inside the notebook.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Infer project root by walking up until we find the big_data_assignment folder
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "big_data_assignment" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)


## 2. Load config and data helpers

Here we:
- Load the central YAML config via `load_config()`.
- Import the shared dataloaders that operate on the CSV files.
- Import the `run_baselines()` pipeline that trains/evaluates the baseline models.

Hyperparameters (e.g. model type and settings) are controlled from `config/config.yaml` under the `model` section.


In [ ]:
from src.utils.config import load_config
from src.data.dataloaders import load_train_data, load_validation_data
from src.pipeline.baseline import run_baselines

config = load_config()
config["model"]


## 3. Run shared baselines on CSV data

This will:
- Load train and validation CSVs using the config-driven dataloaders.
- Build a simple bag-of-words representation.
- Train and evaluate the baseline models defined in `src/pipeline/baseline.py`.

The output is a small accuracy summary in the notebook output.


In [ ]:
run_baselines()
